# Comparaison de modèles VERIS → ATT&CK

Ce notebook entraîne plusieurs modèles de classification multi-label pour prédire les techniques MITRE ATT&CK associées à une capacité VERIS.

**Protocole d'évaluation temporelle :**
- **Entraînement** : mappings enterprise `VERIS 1.3.5 / ATT&CK 9.0`, `1.3.7 / 12.1`, `1.4.0 / 16.1`
- **Test** : mapping `VERIS 1.4.1 / ATT&CK 19.1`

Les métriques comparées incluent la précision (F1, Jaccard, Precision@k) et la consommation de ressources (temps d'entraînement et d'inférence).

In [ ]:
import sqlite3
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MultiLabelBinarizer, normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "databases").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from paths import MAPPINGS_DB, MAPPING_SETS

TRAIN_SETS = MAPPING_SETS[:3]
TEST_SET = MAPPING_SETS[3]
TOP_K = 10
RANDOM_STATE = 42

print(f"Projet : {ROOT}")
print(f"Entraînement : {TRAIN_SETS}")
print(f"Test         : {TEST_SET}")

: 

## 1. Chargement et préparation des données

In [ ]:
def load_mappings(db_path: Path, veris_version: str, attack_version: str) -> pd.DataFrame:
    query = """
        SELECT
            m.capability_id,
            m.capability_description,
            m.capability_group,
            m.attack_object_id,
            m.attack_object_name
        FROM mappings m
        JOIN mapping_sets ms ON m.mapping_set_id = ms.id
        WHERE ms.mapping_framework_version = ?
          AND ms.attack_version = ?
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn, params=(veris_version, attack_version))
    df["capability_key"] = df["capability_id"].str.lower()
    return df


def aggregate_by_capability(df: pd.DataFrame) -> pd.DataFrame:
    """Regroupe les techniques ATT&CK par capacité VERIS."""
    grouped = (
        df.groupby("capability_key", as_index=False)
        .agg(
            capability_id=("capability_id", "first"),
            capability_description=("capability_description", "first"),
            capability_group=("capability_group", "first"),
            attack_object_ids=("attack_object_id", lambda s: sorted(set(s))),
        )
    )
    grouped["text"] = (
        grouped["capability_group"].fillna("")
        + " "
        + grouped["capability_id"].fillna("")
        + " "
        + grouped["capability_description"].fillna("")
    ).str.strip()
    return grouped


train_frames = [
    load_mappings(MAPPINGS_DB, veris, attack)
    for attack, veris, _ in TRAIN_SETS
]
train_raw = pd.concat(train_frames, ignore_index=True)

test_attack, test_veris, _ = TEST_SET
test_raw = load_mappings(MAPPINGS_DB, test_veris, test_attack)

train_df = aggregate_by_capability(train_raw)
test_df = aggregate_by_capability(test_raw)

# Espace d'étiquettes = techniques ATT&CK présentes dans le jeu de test (19.1)
label_space = sorted(test_df["attack_object_ids"].explode().dropna().unique())


def filter_labels(techniques: list[str]) -> list[str]:
    return [t for t in techniques if t in label_space]


train_df["labels"] = train_df["attack_object_ids"].apply(filter_labels)
test_df["labels"] = test_df["attack_object_ids"].apply(filter_labels)

# Retirer les exemples sans étiquette dans l'espace cible
train_df = train_df[train_df["labels"].map(len) > 0].reset_index(drop=True)
test_df = test_df[test_df["labels"].map(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer(classes=label_space)
y_train = mlb.fit_transform(train_df["labels"])
y_test = mlb.transform(test_df["labels"])
X_train_text = train_df["text"].tolist()
X_test_text = test_df["text"].tolist()

overlap = set(train_df["capability_key"]) & set(test_df["capability_key"])

print(f"Exemples d'entraînement : {len(train_df)}")
print(f"Exemples de test        : {len(test_df)}")
print(f"Capacités communes        : {len(overlap)}")
print(f"Capacités test uniquement : {len(test_df) - len(overlap)}")
print(f"Espace d'étiquettes (ATT&CK 19.1) : {len(label_space)} techniques")
print(f"Moyenne techniques / capacité (test) : {y_test.sum(axis=1).mean():.1f}")

train_df.head(3)

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=5000,
    sublinear_tf=True,
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Matrice TF-IDF : {X_train.shape} -> {X_test.shape}")

## 2. Fonctions d'évaluation

In [ ]:
def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int = TOP_K) -> float:
    """Precision@k moyenne sur tous les exemples de test."""
    precisions = []
    for i in range(y_true.shape[0]):
        true_idx = set(np.where(y_true[i] == 1)[0])
        if not true_idx:
            continue
        top_k = np.argsort(scores[i])[::-1][:k]
        pred_idx = set(top_k)
        precisions.append(len(true_idx & pred_idx) / k)
    return float(np.mean(precisions)) if precisions else 0.0


def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int = TOP_K) -> float:
    recalls = []
    for i in range(y_true.shape[0]):
        true_idx = set(np.where(y_true[i] == 1)[0])
        if not true_idx:
            continue
        top_k = np.argsort(scores[i])[::-1][:k]
        pred_idx = set(top_k)
        recalls.append(len(true_idx & pred_idx) / len(true_idx))
    return float(np.mean(recalls)) if recalls else 0.0


def evaluate_model(name: str, y_pred: np.ndarray, y_scores: np.ndarray, train_s: float, infer_s: float) -> dict:
    return {
        "modèle": name,
        "subset_accuracy": accuracy_score(y_test, y_pred),
        "f1_micro": f1_score(y_test, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_samples": f1_score(y_test, y_pred, average="samples", zero_division=0),
        "jaccard_samples": jaccard_score(y_test, y_pred, average="samples", zero_division=0),
        "hamming_loss": hamming_loss(y_test, y_pred),
        f"precision@{TOP_K}": precision_at_k(y_test, y_scores, TOP_K),
        f"recall@{TOP_K}": recall_at_k(y_test, y_scores, TOP_K),
        "train_time_s": train_s,
        "infer_time_s": infer_s,
        "infer_ms_per_sample": (infer_s / len(y_test)) * 1000,
    }


def scores_from_estimator(model, X) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


results: list[dict] = []

## 3. Entraînement et évaluation des modèles

In [ ]:
def fit_and_evaluate(name: str, estimator) -> dict:
    print(f"Entraînement : {name}...")
    t0 = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = estimator.predict(X_test)
    y_scores = scores_from_estimator(estimator, X_test)
    infer_s = time.perf_counter() - t1

    metrics = evaluate_model(name, y_pred, y_scores, train_s, infer_s)
    results.append(metrics)
    print(
        f"  F1 micro={metrics['f1_micro']:.3f} | "
        f"Jaccard={metrics['jaccard_samples']:.3f} | "
        f"P@{TOP_K}={metrics[f'precision@{TOP_K}']:.3f} | "
        f"train={train_s:.1f}s infer={infer_s:.2f}s"
    )
    return metrics


# --- Modèle 1 : Régression logistique (One-vs-Rest) ---
fit_and_evaluate(
    "Logistic Regression (OvR)",
    OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        n_jobs=-1,
    ),
)

# --- Modèle 2 : SVM linéaire (One-vs-Rest) ---
fit_and_evaluate(
    "Linear SVM (OvR)",
    OneVsRestClassifier(
        LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
        n_jobs=-1,
    ),
)

# --- Modèle 3 : Naive Bayes multinomial (One-vs-Rest) ---
fit_and_evaluate(
    "Multinomial Naive Bayes (OvR)",
    OneVsRestClassifier(MultinomialNB(), n_jobs=-1),
)

# --- Modèle 4 : SGDClassifier (rapide, scalable) ---
fit_and_evaluate(
    "SGD Classifier (OvR)",
    OneVsRestClassifier(
        SGDClassifier(loss="log_loss", class_weight="balanced", random_state=RANDOM_STATE),
        n_jobs=-1,
    ),
)

In [ ]:
# --- Modèle 5 : k-NN sur similarité TF-IDF (approche retrieval) ---
print("Entraînement : k-NN Retrieval (TF-IDF)...")

X_train_norm = normalize(X_train)
X_test_norm = normalize(X_test)

t0 = time.perf_counter()
knn = KNeighborsClassifier(n_neighbors=5, metric="cosine", n_jobs=-1)
knn.fit(X_train_norm, y_train)
train_s = time.perf_counter() - t0

t1 = time.perf_counter()
neighbor_indices = knn.kneighbors(X_test_norm, return_distance=False)

y_scores_knn = np.zeros_like(y_test, dtype=float)
for i, neighbors in enumerate(neighbor_indices):
    y_scores_knn[i] = y_train[neighbors].mean(axis=0)

y_pred_knn = (y_scores_knn >= 0.2).astype(int)
infer_s = time.perf_counter() - t1

metrics_knn = evaluate_model("k-NN Retrieval (TF-IDF)", y_pred_knn, y_scores_knn, train_s, infer_s)
results.append(metrics_knn)
print(
    f"  F1 micro={metrics_knn['f1_micro']:.3f} | "
    f"Jaccard={metrics_knn['jaccard_samples']:.3f} | "
    f"P@{TOP_K}={metrics_knn[f'precision@{TOP_K}']:.3f} | "
    f"train={train_s:.1f}s infer={infer_s:.2f}s"
)

In [ ]:
# --- Modèle 6 (optionnel) : k-NN avec descriptions ATT&CK enrichies ---
ATTACK_DB = ROOT / "data" / "databases" / "attack_vocabulary.db"

if ATTACK_DB.exists():
    query = """
        SELECT external_id, name, description
        FROM techniques t
        JOIN attack_versions av ON t.attack_version_id = av.id
        WHERE av.version = ?
    """
    with sqlite3.connect(ATTACK_DB) as conn:
        tech_df = pd.read_sql_query(query, conn, params=(test_attack,))

    tech_text = {
        row.external_id: f"{row.name} {row.description or ''}".strip()
        for row in tech_df.itertuples()
    }

    train_enriched = [
        text + " " + " ".join(tech_text.get(t, "") for t in labels)
        for text, labels in zip(train_df["text"], train_df["labels"])
    ]
    # Au test, on utilise uniquement le texte VERIS (pas les étiquettes ATT&CK)
    test_enriched_infer = test_df["text"].tolist()

    vec_enriched = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=8000, sublinear_tf=True)
    X_train_e = normalize(vec_enriched.fit_transform(train_enriched))
    X_test_e = normalize(vec_enriched.transform(test_enriched_infer))

    print("Entraînement : k-NN Enriched (VERIS + ATT&CK descriptions)...")
    t0 = time.perf_counter()
    knn_e = KNeighborsClassifier(n_neighbors=5, metric="cosine", n_jobs=-1)
    knn_e.fit(X_train_e, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    neighbors = knn_e.kneighbors(X_test_e, return_distance=False)
    y_scores_e = np.zeros_like(y_test, dtype=float)
    for i, idxs in enumerate(neighbors):
        y_scores_e[i] = y_train[idxs].mean(axis=0)
    y_pred_e = (y_scores_e >= 0.2).astype(int)
    infer_s = time.perf_counter() - t1

    metrics_e = evaluate_model(
        "k-NN Enriched (VERIS+ATT&CK)", y_pred_e, y_scores_e, train_s, infer_s
    )
    results.append(metrics_e)
    print(
        f"  F1 micro={metrics_e['f1_micro']:.3f} | "
        f"Jaccard={metrics_e['jaccard_samples']:.3f} | "
        f"P@{TOP_K}={metrics_e[f'precision@{TOP_K}']:.3f}"
    )
else:
    print("attack_vocabulary.db introuvable — modèle enrichi ignoré.")

## 4. Comparaison finale

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1_micro", ascending=False)
display_cols = [
    "modèle",
    "f1_micro",
    "f1_macro",
    "jaccard_samples",
    f"precision@{TOP_K}",
    f"recall@{TOP_K}",
    "hamming_loss",
    "train_time_s",
    "infer_ms_per_sample",
]
results_df[display_cols].round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

order = results_df["modèle"].tolist()

sns.barplot(data=results_df, y="modèle", x="f1_micro", order=order, ax=axes[0], palette="Blues_r")
axes[0].set_title("F1 micro (précision globale)")
axes[0].set_xlabel("F1 micro")

sns.barplot(
    data=results_df,
    y="modèle",
    x=f"precision@{TOP_K}",
    order=order,
    ax=axes[1],
    palette="Greens_r",
)
axes[1].set_title(f"Precision@{TOP_K}")
axes[1].set_xlabel(f"Precision@{TOP_K}")

sns.barplot(
    data=results_df,
    y="modèle",
    x="train_time_s",
    order=order,
    ax=axes[2],
    palette="Oranges_r",
)
axes[2].set_title("Temps d'entraînement (s)")
axes[2].set_xlabel("Secondes")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    results_df["train_time_s"],
    results_df["f1_micro"],
    s=120,
    c=results_df[f"precision@{TOP_K}"],
    cmap="viridis",
    edgecolors="black",
)
for _, row in results_df.iterrows():
    ax.annotate(
        row["modèle"].split("(")[0].strip(),
        (row["train_time_s"], row["f1_micro"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8,
    )
ax.set_xlabel("Temps d'entraînement (s)")
ax.set_ylabel("F1 micro")
ax.set_title("Compromis précision / ressources")
plt.colorbar(scatter, label=f"Precision@{TOP_K}")
plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"\nMeilleur modèle (F1 micro) : {best['modèle']}")
print(f"  F1 micro      : {best['f1_micro']:.4f}")
print(f"  Jaccard       : {best['jaccard_samples']:.4f}")
print(f"  Precision@{TOP_K} : {best[f'precision@{TOP_K}']:.4f}")
print(f"  Train         : {best['train_time_s']:.2f}s | Infer : {best['infer_ms_per_sample']:.1f} ms/échantillon")

## 5. Analyse détaillée — phase de test

Cette section se concentre sur les **résultats bruts** du meilleur modèle sur le jeu de test
(`VERIS 1.4.1 / ATT&CK 19.1`). On décompose la précision étiquette par étiquette,
capacité par capacité, et on distingue les capacités déjà vues à l'entraînement de celles
qui sont nouvelles.

In [ ]:
BEST_MODEL_NAME = results_df.iloc[0]["modèle"]
print(f"Modèle retenu pour l'analyse : {BEST_MODEL_NAME}")

best_model = OneVsRestClassifier(
    LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
    n_jobs=-1,
)
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)
y_scores_best = best_model.decision_function(X_test)


def multilabel_confusion(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Comptages globaux sur toutes les paires (capacité, technique)."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}


def raw_precision_recall_f1(counts: dict) -> dict:
    tp, fp, fn = counts["TP"], counts["FP"], counts["FN"]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "précision": precision,
        "rappel": recall,
        "f1": f1,
    }


def per_sample_metrics(y_true_row, y_pred_row, y_scores_row, k=TOP_K) -> dict:
    true_idx = set(np.where(y_true_row == 1)[0])
    pred_idx = set(np.where(y_pred_row == 1)[0])
    top_k_idx = set(np.argsort(y_scores_row)[::-1][:k])

    tp = len(true_idx & pred_idx)
    fp = len(pred_idx - true_idx)
    fn = len(true_idx - pred_idx)
    hits_at_k = len(true_idx & top_k_idx)

    precision = tp / (tp + fp) if (tp + fp) else (1.0 if not true_idx else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    p_at_k = hits_at_k / k
    r_at_k = hits_at_k / len(true_idx) if true_idx else 1.0
    jaccard = len(true_idx & pred_idx) / len(true_idx | pred_idx) if (true_idx | pred_idx) else 1.0

    return {
        "nb_vraies": len(true_idx),
        "nb_prédites": len(pred_idx),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "correctes_top_k": hits_at_k,
        f"précision": precision,
        f"rappel": recall,
        f"precision@{k}": p_at_k,
        f"recall@{k}": r_at_k,
        "jaccard": jaccard,
    }


global_counts = multilabel_confusion(y_test, y_pred_best)
global_prf = raw_precision_recall_f1(global_counts)

print("\n=== Comptages bruts (niveau étiquette) ===")
for k, v in global_counts.items():
    print(f"  {k:2} : {v:,}")

print("\n=== Précision / rappel / F1 bruts (micro) ===")
for k, v in global_prf.items():
    print(f"  {k:10} : {v:.4f} ({v*100:.1f} %)")

total_predictions = global_counts["TP"] + global_counts["FP"]
total_truth = global_counts["TP"] + global_counts["FN"]
print(f"\n  Prédictions positives émises : {total_predictions:,}")
print(f"  Associations vraies à retrouver : {total_truth:,}")
print(f"  Taux de bonnes prédictions (TP / total prédit) : {global_counts['TP']/total_predictions:.1%}")
print(f"  Taux de couverture (TP / total vrai)           : {global_counts['TP']/total_truth:.1%}")

In [ ]:
# --- Précision@k pour différentes valeurs de k ---
k_values = list(range(1, 21))
pk_rows = []
for k in k_values:
    p = precision_at_k(y_test, y_scores_best, k)
    r = recall_at_k(y_test, y_scores_best, k)
    pk_rows.append({"k": k, "precision@k": p, "recall@k": r, "f1@k": 2 * p * r / (p + r) if (p + r) else 0})

pk_df = pd.DataFrame(pk_rows)
print("=== Précision et rappel@k (moyenne sur le test) ===")
display(pk_df.round(4))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(pk_df["k"], pk_df["precision@k"], marker="o", label="Precision@k", color="#2ecc71")
ax1.plot(pk_df["k"], pk_df["recall@k"], marker="s", label="Recall@k", color="#3498db")
ax1.axvline(TOP_K, color="gray", linestyle="--", alpha=0.7, label=f"k={TOP_K} (seuil notebook)")
ax1.set_xlabel("k (nombre de suggestions)")
ax1.set_ylabel("Score")
ax1.set_title(f"Évolution Precision@k / Recall@k — {BEST_MODEL_NAME}")
ax1.set_xticks(k_values)
ax1.set_ylim(0, 1)
ax1.legend()
plt.tight_layout()
plt.show()

row_k10 = pk_df[pk_df["k"] == TOP_K].iloc[0]
print(f"\nÀ k={TOP_K} : {row_k10['precision@k']*100:.1f} % des suggestions sont correctes")
print(f"À k={TOP_K} : {row_k10['recall@k']*100:.1f} % des vraies techniques sont retrouvées")

In [ ]:
# --- Résultats par capacité VERIS (176 lignes de test) ---
test_results = []
for i in range(len(test_df)):
    m = per_sample_metrics(y_test[i], y_pred_best[i], y_scores_best[i], k=TOP_K)
    true_labels = [mlb.classes_[j] for j in np.where(y_test[i] == 1)[0]]
    pred_labels = [mlb.classes_[j] for j in np.where(y_pred_best[i] == 1)[0]]
    top_k_labels = [mlb.classes_[j] for j in np.argsort(y_scores_best[i])[::-1][:TOP_K]]
    correct_top_k = sorted(set(true_labels) & set(top_k_labels))
    missed = sorted(set(true_labels) - set(top_k_labels))
    false_pos = sorted(set(top_k_labels) - set(true_labels))

    test_results.append({
        "capability_id": test_df.loc[i, "capability_id"],
        "description": test_df.loc[i, "capability_description"],
        "vue_entraînement": "vue" if test_df.loc[i, "capability_key"] in overlap else "nouvelle",
        **m,
        "techniques_correctes_top_k": correct_top_k,
        "techniques_manquées": missed,
        "fausses_suggestions_top_k": false_pos,
    })

test_results_df = pd.DataFrame(test_results)

print("=== Statistiques par capacité (test) ===")
summary_cols = ["nb_vraies", "nb_prédites", "tp", "fp", "fn", "correctes_top_k",
                "précision", "rappel", f"precision@{TOP_K}", f"recall@{TOP_K}", "jaccard"]
print(test_results_df[summary_cols].describe().round(3).T)

print(f"\nCapacités avec precision@{TOP_K} = 100 % : {(test_results_df[f'precision@{TOP_K}'] == 1).sum()} / {len(test_results_df)}")
print(f"Capacités avec recall@{TOP_K} = 100 %    : {(test_results_df[f'recall@{TOP_K}'] == 1).sum()} / {len(test_results_df)}")
print(f"Capacités avec jaccard = 0 (aucun match) : {(test_results_df['jaccard'] == 0).sum()} / {len(test_results_df)}")

test_results_df[summary_cols + ["capability_id", "vue_entraînement"]].sort_values(
    f"precision@{TOP_K}", ascending=False
).head(10)

In [ ]:
# --- Capacités vues vs nouvelles à l'entraînement ---
seen_df = test_results_df[test_results_df["vue_entraînement"] == "vue"]
new_df = test_results_df[test_results_df["vue_entraînement"] == "nouvelle"]

compare_rows = []
for label, subset in [("Vue à l'entraînement", seen_df), ("Nouvelle au test", new_df), ("Total test", test_results_df)]:
    compare_rows.append({
        "groupe": label,
        "n_capacités": len(subset),
        "précision_moyenne": subset["précision"].mean(),
        "rappel_moyen": subset["rappel"].mean(),
        f"precision@{TOP_K}_moyenne": subset[f"precision@{TOP_K}"].mean(),
        f"recall@{TOP_K}_moyen": subset[f"recall@{TOP_K}"].mean(),
        "jaccard_moyen": subset["jaccard"].mean(),
        "tp_total": subset["tp"].sum(),
        "fp_total": subset["fp"].sum(),
        "fn_total": subset["fn"].sum(),
    })

compare_df = pd.DataFrame(compare_rows)
print("=== Précision brute par groupe de capacités ===")
display(compare_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=test_results_df, x="vue_entraînement", y=f"precision@{TOP_K}", ax=axes[0], palette="Set2")
axes[0].set_title(f"Precision@{TOP_K} par capacité")
axes[0].set_xlabel("")

sns.boxplot(data=test_results_df, x="vue_entraînement", y="jaccard", ax=axes[1], palette="Set2")
axes[1].set_title("Jaccard par capacité")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

In [ ]:
# --- Distribution de la précision sur le test ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(test_results_df[f"precision@{TOP_K}"], bins=20, color="#2ecc71", edgecolor="white")
axes[0].axvline(test_results_df[f"precision@{TOP_K}"].mean(), color="red", linestyle="--",
                label=f"moyenne = {test_results_df[f'precision@{TOP_K}'].mean():.2f}")
axes[0].set_title(f"Distribution Precision@{TOP_K}")
axes[0].set_xlabel("Precision@k")
axes[0].legend()

axes[1].hist(test_results_df["nb_vraies"], bins=15, color="#3498db", edgecolor="white", alpha=0.8, label="Vraies")
axes[1].hist(test_results_df["correctes_top_k"], bins=15, color="#2ecc71", edgecolor="white", alpha=0.6, label="Correctes (top k)")
axes[1].set_title("Techniques par capacité")
axes[1].set_xlabel("Nombre de techniques")
axes[1].legend()

axes[2].scatter(test_results_df["nb_vraies"], test_results_df[f"precision@{TOP_K}"], alpha=0.6, c="#9b59b6")
axes[2].set_xlabel("Nb vraies techniques")
axes[2].set_ylabel(f"Precision@{TOP_K}")
axes[2].set_title("Précision vs complexité du mapping")

plt.tight_layout()
plt.show()

In [ ]:
def show_prediction_detail(row_idx: int) -> None:
    """Affiche une prédiction exemple avec détail TP / FP / FN."""
    row = test_results_df.iloc[row_idx]
    print(f"Capacité  : {row['capability_id']}")
    print(f"Description : {row['description']}")
    print(f"Statut    : {row['vue_entraînement']} à l'entraînement")
    print(f"Vraies ({row['nb_vraies']}) : {row['techniques_correctes_top_k'] + row['techniques_manquées']}")
    print(f"Top-{TOP_K} prédit : {row['techniques_correctes_top_k'] + row['fausses_suggestions_top_k']}")
    print(f"  ✓ Correctes  ({row['correctes_top_k']}) : {row['techniques_correctes_top_k']}")
    print(f"  ✗ Manquées   ({len(row['techniques_manquées'])}) : {row['techniques_manquées']}")
    print(f"  ✗ Fausses    ({len(row['fausses_suggestions_top_k'])}) : {row['fausses_suggestions_top_k']}")
    print(f"  → Precision@{TOP_K} = {row[f'precision@{TOP_K}']:.2f} | Recall@{TOP_K} = {row[f'recall@{TOP_K}']:.2f} | Jaccard = {row['jaccard']:.2f}")


print("=" * 70)
print("MEILLEURE PRÉDICTION")
print("=" * 70)
best_idx = test_results_df[f"precision@{TOP_K}"].idxmax()
show_prediction_detail(best_idx)

print("\n" + "=" * 70)
print("PRÉDICTION MÉDIANE")
print("=" * 70)
median_idx = test_results_df[f"precision@{TOP_K}"].sub(test_results_df[f"precision@{TOP_K}"].median()).abs().idxmin()
show_prediction_detail(median_idx)

print("\n" + "=" * 70)
print("PIRE PRÉDICTION")
print("=" * 70)
worst_idx = test_results_df[f"precision@{TOP_K}"].idxmin()
show_prediction_detail(worst_idx)

In [ ]:
# --- Précision par technique ATT&CK (589 labels) ---
tech_stats = []
for j, tech_id in enumerate(mlb.classes_):
    y_t = y_test[:, j]
    y_p = y_pred_best[:, j]
    tp = int(np.sum((y_t == 1) & (y_p == 1)))
    fp = int(np.sum((y_t == 0) & (y_p == 1)))
    fn = int(np.sum((y_t == 1) & (y_p == 0)))
    support = int(y_t.sum())
    if support == 0:
        continue
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    tech_stats.append({
        "technique": tech_id,
        "support_test": support,
        "tp": tp, "fp": fp, "fn": fn,
        "précision": prec,
        "rappel": rec,
        "f1": 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0,
    })

tech_stats_df = pd.DataFrame(tech_stats).sort_values("support_test", ascending=False)

print(f"=== Techniques les plus fréquentes dans le test (top 10) ===")
display(tech_stats_df.head(10).round(3))

print(f"\n=== Techniques jamais prédites correctement (FN > 0, TP = 0, support ≥ 2) ===")
never_found = tech_stats_df[(tech_stats_df["tp"] == 0) & (tech_stats_df["support_test"] >= 2)]
print(f"{len(never_found)} techniques sur {len(tech_stats_df)}")
display(never_found.head(10)[["technique", "support_test", "fn"]])

print(f"\n=== Techniques toujours bien prédites (rappel = 100 %, support ≥ 2) ===")
perfect = tech_stats_df[(tech_stats_df["rappel"] == 1.0) & (tech_stats_df["support_test"] >= 2)]
print(f"{len(perfect)} techniques")
display(perfect.head(10)[["technique", "support_test", "tp", "précision"]])

In [ ]:
# --- Tableau récapitulatif final (phase de test) ---
print("=" * 60)
print("RÉCAPITULATIF — PHASE DE TEST")
print(f"Modèle : {BEST_MODEL_NAME}")
print(f"Jeu    : VERIS {test_veris} → ATT&CK {test_attack}")
print(f"Capacités testées : {len(test_df)}")
print("=" * 60)

recap = {
    "Métrique": [
        "Précision brute (TP / TP+FP)",
        "Rappel brut (TP / TP+FN)",
        "F1 micro",
        f"Precision@{TOP_K} (moyenne)",
        f"Recall@{TOP_K} (moyenne)",
        "Jaccard moyen par capacité",
        "Capacités parfaites (P@k=100 %)",
        "Capacités sans aucun match",
        "TP / FP / FN (total étiquettes)",
    ],
    "Valeur": [
        f"{global_prf['précision']:.1%}",
        f"{global_prf['rappel']:.1%}",
        f"{results_df.iloc[0]['f1_micro']:.1%}",
        f"{test_results_df[f'precision@{TOP_K}'].mean():.1%}",
        f"{test_results_df[f'recall@{TOP_K}'].mean():.1%}",
        f"{test_results_df['jaccard'].mean():.1%}",
        f"{(test_results_df[f'precision@{TOP_K}'] == 1).sum()} / {len(test_results_df)}",
        f"{(test_results_df['jaccard'] == 0).sum()} / {len(test_results_df)}",
        f"{global_counts['TP']} / {global_counts['FP']} / {global_counts['FN']}",
    ],
}
pd.DataFrame(recap)

## 6. Notes méthodologiques

- **Tâche** : classification multi-label — chaque capacité VERIS est associée à plusieurs techniques ATT&CK.
- **Features** : texte concaténé (`capability_group` + `capability_id` + `capability_description`), vectorisé en TF-IDF.
- **Split temporel** : les 3 premières versions servent à l'entraînement, la dernière (19.1) au test — simule la généralisation vers une nouvelle version du framework.
- **Précision brute** : `TP / (TP + FP)` — parmi toutes les techniques prédites, quelle proportion est correcte ?
- **Precision@k** : pour chaque capacité, combien des *k* premières suggestions sont correctes, puis moyenne sur le test.
- **Limites** : les IDs ATT&CK évoluent entre versions ; seules les techniques présentes dans ATT&CK 19.1 sont conservées comme étiquettes cibles.